# Coffee Crop Health Monitor — GEE + Gradio Demo
Auto-fetch soil readings from Google Earth Engine (SoilGrids) using GPS coordinates.
Farmer enters location → GEE pulls soil data → Swin-T + XGBoost → Health report.


## Step 1 — Install & Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'gradio', 'earthengine-api', '-q'])

import ee
import gradio as gr
import torch, torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
import pickle

print('All packages ready')


Mounted at /content/drive
All packages ready


## Step 2 — Authenticate Google Earth Engine

In [2]:
# Run this cell once after GEE account is approved.
# It will open a browser tab asking you to authorise with your Google account.

ee.Authenticate()
ee.Initialize(project='coffee-crop-nutrient')   # Replace with your GEE project ID

print('GEE authenticated and initialised')


GEE authenticated and initialised


## Step 3 — Load Both Models

In [3]:
SOIL_PATH = '/content/drive/MyDrive/CODMAV_Internship_Report/Models/SoilAgent/soil_agent_v2.pkl'
LEAF_PATH  = '/content/drive/MyDrive/CODMAV_Internship_Report/Models/LeafAgent/leaf_agent_swin_t.pth'
META_PATH  = '/content/drive/MyDrive/CODMAV_Internship_Report/Models/LeafAgent/leaf_agent_metadata.pkl'

with open(SOIL_PATH, 'rb') as f:
    soil_bundle = pickle.load(f)

soil_model  = soil_bundle['model']
soil_scaler = soil_bundle['scaler']
SOIL_FEATURES = ['N','P','K','pH','EC','OC','S','Zn','Fe','Cu','Mn','B']
SOIL_CLASSES  = {0:'Low Fertility', 1:'Medium Fertility', 2:'High Fertility'}

with open(META_PATH, 'rb') as f:
    meta = pickle.load(f)

CLASS_NAMES = meta['class_names']
MEAN = meta.get('mean', [0.485, 0.456, 0.406])
STD  = meta.get('std',  [0.229, 0.224, 0.225])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

leaf_model = models.swin_t(weights=None)
leaf_model.head = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(leaf_model.head.in_features, 512),
    nn.GELU(),
    nn.Dropout(p=0.15),
    nn.Linear(512, 9)
)
leaf_model.load_state_dict(torch.load(LEAF_PATH, map_location=device))
leaf_model = leaf_model.to(device)
leaf_model.eval()

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print(f'Soil Agent  : {type(soil_model).__name__}')
print(f'Leaf Agent  : Swin-T  (97.58% val accuracy)')
print(f'Device      : {device}')


Soil Agent  : CalibratedClassifierCV
Leaf Agent  : Swin-T  (97.58% val accuracy)
Device      : cuda


## Step 4 — GEE Soil Fetch Function

In [4]:
# import requests

# REGIONAL_DEFAULTS = {
#     'Chikkamagaluru': {'P':42, 'K':185, 'EC':0.7, 'S':14, 'Fe':9.2, 'Cu':0.6, 'Mn':3.2, 'B':0.35, 'Zn':1.4},
#     'Kodagu':         {'P':48, 'K':210, 'EC':0.9, 'S':16, 'Fe':7.8, 'Cu':0.5, 'Mn':2.9, 'B':0.42, 'Zn':1.6},
#     'Hassan':         {'P':40, 'K':195, 'EC':0.8, 'S':13, 'Fe':8.5, 'Cu':0.4, 'Mn':3.0, 'B':0.38, 'Zn':1.3},
#     'Wayanad':        {'P':50, 'K':220, 'EC':0.75,'S':17, 'Fe':8.0, 'Cu':0.55,'Mn':3.5, 'B':0.45, 'Zn':1.7},
#     'Nilgiris':       {'P':45, 'K':200, 'EC':0.85,'S':15, 'Fe':7.5, 'Cu':0.5, 'Mn':2.8, 'B':0.40, 'Zn':1.5},
#     'Yercaud':        {'P':38, 'K':180, 'EC':0.7, 'S':13, 'Fe':7.0, 'Cu':0.45,'Mn':2.6, 'B':0.36, 'Zn':1.3},
#     'Araku Valley':   {'P':44, 'K':190, 'EC':0.8, 'S':14, 'Fe':8.2, 'Cu':0.5, 'Mn':3.1, 'B':0.38, 'Zn':1.4},
# }
# def fetch_soil_from_gee(lat, lon):
#     url = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
#     params = {
#         'lon': lon, 'lat': lat,
#         'property': ['nitrogen', 'phh2o', 'soc'],
#         'depth': '0-5cm', 'value': 'mean'
#     }
#     N, pH, OC = 120.0, 6.2, 1.2
#     sat_ok = False
#     try:
#         response = requests.get(url, params=params, timeout=15)
#         response.raise_for_status()
#         data = response.json()
#         def extract(prop_name):
#             for layer in data.get('properties', {}).get('layers', []):
#                 if layer['name'] == prop_name:
#                     return layer['depths'][0]['values'].get('mean')
#             return None
#         n_raw  = extract('nitrogen')
#         ph_raw = extract('phh2o')
#         oc_raw = extract('soc')
#         if n_raw  is not None: N  = round(n_raw  * 10, 1)
#         if ph_raw is not None: pH = round(ph_raw / 10, 2)
#         if oc_raw is not None: OC = round(oc_raw / 10, 2)
#         sat_ok = all(v is not None for v in [n_raw, ph_raw, oc_raw])
#         print(f'SoilGrids {"success" if sat_ok else "partial"}: N={N}  pH={pH}  OC={OC}')
#     except Exception as e:
#         print(f'SoilGrids API failed: {e} — using Karnataka defaults')

#     readings = {
#         'N': N, 'P': REGIONAL_DEFAULTS['P'], 'K': REGIONAL_DEFAULTS['K'],
#         'pH': pH, 'EC': REGIONAL_DEFAULTS['EC'], 'OC': OC,
#         'S': REGIONAL_DEFAULTS['S'], 'Zn': REGIONAL_DEFAULTS['Zn'],
#         'Fe': REGIONAL_DEFAULTS['Fe'], 'Cu': REGIONAL_DEFAULTS['Cu'],
#         'Mn': REGIONAL_DEFAULTS['Mn'], 'B': REGIONAL_DEFAULTS['B'],
#     }
#     sources = {
#         'N':  'satellite' if sat_ok else 'regional default',
#         'pH': 'satellite' if sat_ok else 'regional default',
#         'OC': 'satellite' if sat_ok else 'regional default',
#     }
#     for k in ['P','K','EC','S','Zn','Fe','Cu','Mn','B']:
#         sources[k] = 'regional default'
#     return readings, sources

# print('SoilGrids REST API fetch ready')
# print('Satellite: N, pH, OC  |  Defaults: P, K, EC, S, Zn, Fe, Cu, Mn, B')

import requests

# ── District-specific soil defaults (NBSS&LUP averages for South Indian coffee belt) ──
DISTRICT_DEFAULTS = {
    'Chikkamagaluru': {'P':42,  'K':185, 'EC':0.70, 'S':14, 'Fe':9.2, 'Cu':0.60, 'Mn':3.2, 'B':0.35, 'Zn':1.4},
    'Kodagu':         {'P':48,  'K':210, 'EC':0.90, 'S':16, 'Fe':7.8, 'Cu':0.50, 'Mn':2.9, 'B':0.42, 'Zn':1.6},
    'Hassan':         {'P':40,  'K':195, 'EC':0.80, 'S':13, 'Fe':8.5, 'Cu':0.40, 'Mn':3.0, 'B':0.38, 'Zn':1.3},
    'Wayanad':        {'P':50,  'K':220, 'EC':0.75, 'S':17, 'Fe':8.0, 'Cu':0.55, 'Mn':3.5, 'B':0.45, 'Zn':1.7},
    'Nilgiris':       {'P':45,  'K':200, 'EC':0.85, 'S':15, 'Fe':7.5, 'Cu':0.50, 'Mn':2.8, 'B':0.40, 'Zn':1.5},
    'Yercaud':        {'P':38,  'K':180, 'EC':0.70, 'S':13, 'Fe':7.0, 'Cu':0.45, 'Mn':2.6, 'B':0.36, 'Zn':1.3},
    'Araku Valley':   {'P':44,  'K':190, 'EC':0.80, 'S':14, 'Fe':8.2, 'Cu':0.50, 'Mn':3.1, 'B':0.38, 'Zn':1.4},
    'default':        {'P':45,  'K':200, 'EC':0.80, 'S':15, 'Fe':8.0, 'Cu':0.50, 'Mn':3.0, 'B':0.40, 'Zn':1.5},
}

def get_district_defaults(region_name):
    """Returns district-specific soil defaults for a given region name."""
    if region_name:
        for key in DISTRICT_DEFAULTS:
            if key.lower() in str(region_name).lower():
                print(f'Using district defaults for: {key}')
                return DISTRICT_DEFAULTS[key]
    print('Using Karnataka general defaults')
    return DISTRICT_DEFAULTS['default']


def fetch_soil_from_gee(lat, lon, region_name='default'):
    """
    Fetches N, pH, OC from ISRIC SoilGrids REST API (no GEE account required).
    P, K, EC, S, Zn, Fe, Cu, Mn, B use district-specific soil survey averages.

    Parameters
    ----------
    lat, lon      : GPS coordinates
    region_name   : detected coffee region name (used to select district defaults)

    Satellite-derived : N, pH, OC  (SoilGrids 250m v2.0)
    District defaults : P, K, EC, S, Zn, Fe, Cu, Mn, B  (NBSS&LUP district averages)
    """
    url = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
    params = {
        'lon'     : lon,
        'lat'     : lat,
        'property': ['nitrogen', 'phh2o', 'soc'],
        'depth'   : '0-5cm',
        'value'   : 'mean'
    }

    # Defaults in case API fails
    N, pH, OC = 120.0, 6.2, 1.2
    sat_ok = False

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        def extract(prop_name):
            for layer in data.get('properties', {}).get('layers', []):
                if layer['name'] == prop_name:
                    return layer['depths'][0]['values'].get('mean')
            return None

        n_raw  = extract('nitrogen')   # cg/kg  → mg/kg  (× 10)
        ph_raw = extract('phh2o')      # pH×10  → pH     (÷ 10)
        oc_raw = extract('soc')        # dg/kg  → %      (÷ 10)

        if n_raw  is not None: N  = round(n_raw  * 10, 1)
        if ph_raw is not None: pH = round(ph_raw / 10, 2)
        if oc_raw is not None: OC = round(oc_raw / 10, 2)

        sat_ok = all(v is not None for v in [n_raw, ph_raw, oc_raw])
        status = 'success' if sat_ok else 'partial (some values defaulted)'
        print(f'SoilGrids REST API {status}')
        print(f'  N={N} mg/kg  |  pH={pH}  |  OC={OC}%')

    except Exception as e:
        print(f'SoilGrids REST API failed: {e}')
        print('Using district soil survey averages for all parameters.')

    # Get district-specific defaults for P, K, EC, S, Zn, Fe, Cu, Mn, B
    d = get_district_defaults(region_name)

    readings = {
        'N'  : N,
        'P'  : d['P'],
        'K'  : d['K'],
        'pH' : pH,
        'EC' : d['EC'],
        'OC' : OC,
        'S'  : d['S'],
        'Zn' : d['Zn'],
        'Fe' : d['Fe'],
        'Cu' : d['Cu'],
        'Mn' : d['Mn'],
        'B'  : d['B'],
    }

    sources = {
        'N'  : 'satellite' if sat_ok else 'district default',
        'pH' : 'satellite' if sat_ok else 'district default',
        'OC' : 'satellite' if sat_ok else 'district default',
        'P'  : f'district default ({region_name})',
        'K'  : f'district default ({region_name})',
        'EC' : f'district default ({region_name})',
        'S'  : f'district default ({region_name})',
        'Zn' : f'district default ({region_name})',
        'Fe' : f'district default ({region_name})',
        'Cu' : f'district default ({region_name})',
        'Mn' : f'district default ({region_name})',
        'B'  : f'district default ({region_name})',
    }

    return readings, sources


print('SoilGrids REST API + district defaults ready')
print()
print('Satellite (SoilGrids 250m) : N, pH, OC')
print('District soil survey avg   : P, K, EC, S, Zn, Fe, Cu, Mn, B')
print()
print('Districts configured:')
for k in DISTRICT_DEFAULTS:
    if k != 'default':
        d = DISTRICT_DEFAULTS[k]
        print(f'  {k:<18} P={d["P"]}  K={d["K"]}  EC={d["EC"]}')

SoilGrids REST API + district defaults ready

Satellite (SoilGrids 250m) : N, pH, OC
District soil survey avg   : P, K, EC, S, Zn, Fe, Cu, Mn, B

Districts configured:
  Chikkamagaluru     P=42  K=185  EC=0.7
  Kodagu             P=48  K=210  EC=0.9
  Hassan             P=40  K=195  EC=0.8
  Wayanad            P=50  K=220  EC=0.75
  Nilgiris           P=45  K=200  EC=0.85
  Yercaud            P=38  K=180  EC=0.7
  Araku Valley       P=44  K=190  EC=0.8


## Step 5 — Prediction + Fusion Functions

In [5]:
import torch.nn.functional as F

HEALTH_MATRIX = {
    (2,'healthy')  : ('Excellent',          '#2d6a4f'),
    (1,'healthy')  : ('Good',               '#52b788'),
    (0,'healthy')  : ('Monitor Soil',       '#f4a261'),
    (2,'deficient'): ('Nutrient Imbalance', '#e76f51'),
    (1,'deficient'): ('Nutrient Deficiency','#e76f51'),
    (0,'deficient'): ('Critical',           '#c1121f'),
}
CONF_THRESHOLD = 0.60
OOD_THRESHOLD  = None   # set after calibration cell runs

def predict_soil(readings):
    import pandas as pd
    vals   = pd.DataFrame([[readings[f] for f in SOIL_FEATURES]], columns=SOIL_FEATURES)
    scaled = soil_scaler.transform(vals)
    probs  = soil_model.predict_proba(scaled)[0]
    pred   = int(np.argmax(probs))
    return pred, SOIL_CLASSES[pred], round(float(probs[pred])*100,1), probs

import torch.nn.functional as F

# ── Step 1: Build coffee leaf feature reference (run once after loading model) ──
def extract_features(img_tensor):
    """Extract feature vector from Swin-T backbone (before classification head)."""
    features = []
    def hook(module, input, output):
        features.append(output.detach())

    # Hook into the final norm layer before the head
    handle = leaf_model.norm.register_forward_hook(hook)
    with torch.no_grad():
        leaf_model(img_tensor)
    handle.remove()
    # Global average pool → feature vector
    return features[0].mean(dim=1)  # shape: (1, 768)

# Build reference by running a few training images through the model
# We use the test directory since training images are available
import os
from pathlib import Path

# TEST_DIR = '/content/drive/MyDrive/CODMAV_Internship_Report/Datasets/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test'
TEST_DIR = '/content/drive/MyDrive/CODMAV_Internship_Report/DATASETS/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test'

print('Building coffee leaf feature reference...')
ref_features = []
for class_dir in sorted(Path(TEST_DIR).iterdir()):
    if not class_dir.is_dir(): continue
    images = list(class_dir.glob('*.jpg'))[:5]  # 5 per class = 45 total
    for img_path in images:
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = eval_tf(img).unsqueeze(0).to(device)
            feat = extract_features(tensor)
            ref_features.append(feat)
        except: continue

COFFEE_REFERENCE = torch.cat(ref_features, dim=0)  # shape: (N, 768)
COFFEE_MEAN = COFFEE_REFERENCE.mean(dim=0, keepdim=True)  # mean coffee feature
COFFEE_STD  = COFFEE_REFERENCE.std(dim=0).mean().item()   # spread

print(f'Reference built from {len(ref_features)} coffee leaf images')
print(f'Feature mean norm: {COFFEE_MEAN.norm().item():.2f}')
print(f'Feature std: {COFFEE_STD:.4f}')

# ── Step 2: OOD threshold (set after seeing distance distribution) ──
# We will set this after running the calibration cell below
OOD_THRESHOLD = None  # will be set in next cell


# ── Calibrate OOD threshold ─────────────────────────────────────────────────
# Run all reference images through and see their distances
distances = []
for feat in ref_features:
    # Ensure both tensors are 2D (1, 768) to get a single similarity score
    dist = F.cosine_similarity(feat.view(1, -1), COFFEE_MEAN.view(1, -1)).item()
    distances.append(dist)

min_dist = min(distances)
mean_dist = sum(distances)/len(distances)

print(f'Coffee leaf similarity to mean:')
print(f'  Min  : {min_dist:.4f}')
print(f'  Mean : {mean_dist:.4f}')
print()

# Set threshold at slightly below the minimum coffee similarity
# Anything below this = not a coffee leaf
OOD_THRESHOLD = min_dist - 0.05
print(f'OOD threshold set to: {OOD_THRESHOLD:.4f}')
print('Images with cosine similarity below this will be rejected.')

def predict_leaf(img):
    if img is None: return None, None, None, None
    if isinstance(img, np.ndarray): img = Image.fromarray(img)

    inp = eval_tf(img).unsqueeze(0).to(device)

    # -- OOD check --
    feat = extract_features(inp)

    # FIX: Add .view(1, -1) to both tensors here
    similarity = F.cosine_similarity(feat.view(1, -1), COFFEE_MEAN.view(1, -1)).item()

    if OOD_THRESHOLD is not None and similarity < OOD_THRESHOLD:
        print(f'OOD detected: similarity={similarity:.4f} < threshold={OOD_THRESHOLD:.4f}')
        return 'NOT_COFFEE_LEAF', round(similarity*100, 1), None, None

    # -- Normal prediction --
    with torch.no_grad():
        probs = torch.softmax(leaf_model(inp), dim=1).cpu().numpy()[0]
    top3 = probs.argsort()[::-1][:3]
    return CLASS_NAMES[top3[0]], round(float(probs[top3[0]])*100, 1), top3, probs




def build_report(soil_cls, soil_label, soil_conf, soil_probs,
                 leaf_name, leaf_conf, top3, leaf_probs, sources=None):

    # Leaf rejection — stop here, show no soil info
    if leaf_name == 'NOT_COFFEE_LEAF':
        thresh_str = f'{round(OOD_THRESHOLD*100,1)}%' if OOD_THRESHOLD else 'N/A'
        return (
            "INVALID INPUT — Not a Coffee Leaf\n"
            + "="*52 + "\n\n"
            + "  The uploaded image does not appear to be a\n"
            + "  coffee plant leaf.\n\n"
            + f"  Similarity score : {leaf_conf}%\n"
            + f"  Threshold        : {thresh_str}\n\n"
            + "  Please upload a clear, well-lit photo of a\n"
            + "  coffee plant leaf for accurate diagnosis.\n"
            + "="*52
        )

    leaf_type    = 'healthy' if leaf_name == 'healthy' else 'deficient'
    overall, _   = HEALTH_MATRIX.get((soil_cls, leaf_type), ('Unknown','#888'))
    overall_conf = round((soil_conf/100 * leaf_conf/100)**0.5 * 100, 1)
    flags        = []
    if soil_conf < CONF_THRESHOLD*100: flags.append('Soil Agent')
    if leaf_conf < CONF_THRESHOLD*100: flags.append('Leaf Agent')

    top3_str = '  |  '.join(
        f"{CLASS_NAMES[i]}: {leaf_probs[i]*100:.1f}%" for i in top3
    )
    soil_prob_str = '  |  '.join(
        f"{SOIL_CLASSES[i]}: {soil_probs[i]*100:.1f}%" for i in range(3)
    )

    src_note = ''
    if sources:
        sat  = [k for k,v in sources.items() if v == 'satellite']
        dflt = [k for k,v in sources.items() if v == 'regional default']
        src_note = f"""
DATA SOURCES
  Satellite (GEE/SoilGrids) : {', '.join(sat)}
  Regional defaults          : {', '.join(dflt)}
"""

    report = f"""
COFFEE CROP HEALTH ASSESSMENT
{'='*52}

  Overall Health       :  {overall}
  Overall Confidence   :  {overall_conf}%
  Expert Review        :  {'RECOMMENDED — ' + ', '.join(flags) if flags else 'Not required'}

SOIL ANALYSIS
  Fertility            :  {soil_label}
  Confidence           :  {soil_conf}%
  Probabilities        :  {soil_prob_str}

LEAF ANALYSIS
  Deficiency           :  {leaf_name}
  Confidence           :  {leaf_conf}%
  Top 3 predictions    :  {top3_str}
{src_note}
STATUS: {'Low confidence — verify with agronomist' if flags else 'High confidence prediction'}
{'='*52}
""".strip()

    return report

COFFEE_MEAN = None   # built in next cell
print('Predict functions ready')
print('OOD check: PENDING — run calibration cell next')


Building coffee leaf feature reference...
Reference built from 45 coffee leaf images
Feature mean norm: 11.93
Feature std: 0.4206
Coffee leaf similarity to mean:
  Min  : 0.2600
  Mean : 0.3246

OOD threshold set to: 0.2100
Images with cosine similarity below this will be rejected.
Predict functions ready
OOD check: PENDING — run calibration cell next


In [6]:
import os

# Your path
TEST_DIR = '/content/drive/MyDrive/CODMAV_Internship_Report/DATASETS/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test'

if os.path.exists(TEST_DIR):
    print("✅ Path is CORRECT!")
    # List the contents to be 100% sure
    print("Folders found:", os.listdir(TEST_DIR))
else:
    print("❌ Path is INVALID.")
    # Check where it breaks
    if not os.path.exists('/content/drive/MyDrive'):
        print("Note: Your Drive might not be mounted yet. Run Step 1 again.")

✅ Path is CORRECT!
Folders found: ['magnesium-Mg', 'phosphorus-P', 'nitrogen-N', 'manganese-Mn', 'iron-Fe', 'healthy', 'calcium-Ca', 'boron-B', 'potassium-K']


In [7]:
# ── OOD Calibration — build coffee leaf reference features ──────────────────
# Run this ONCE after loading the model. Takes ~30 seconds.

from pathlib import Path

TEST_DIR = '/content/drive/MyDrive/CODMAV_Internship_Report/DATASETS/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test'

print('Building coffee leaf feature reference...')
ref_features = []

for class_dir in sorted(Path(TEST_DIR).iterdir()):
    if not class_dir.is_dir(): continue
    images = list(class_dir.glob('*.jpg'))[:5]  # 5 per class = 45 total
    for img_path in images:
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = eval_tf(img).unsqueeze(0).to(device)
            feat = extract_features(tensor)
            ref_features.append(feat)
        except:
            continue

COFFEE_REFERENCE = torch.cat(ref_features, dim=0)        # (N, 768)
COFFEE_MEAN      = COFFEE_REFERENCE.mean(dim=0, keepdim=True).to(device)  # (1, 768)

# Compute similarity of all reference images to their mean

distances = [
    F.cosine_similarity(f.to(device).view(1, -1), COFFEE_MEAN.view(1, -1)).item()
    for f in ref_features
]
min_sim  = min(distances)
mean_sim = sum(distances) / len(distances)

# Threshold = min similarity minus a small margin
OOD_THRESHOLD = min_sim - 0.05

print(f'Reference built from {len(ref_features)} coffee leaf images')
print(f'  Similarity range : {min_sim:.4f} – {max(distances):.4f}')
print(f'  Mean similarity  : {mean_sim:.4f}')
print(f'  OOD threshold    : {OOD_THRESHOLD:.4f}')
print()
print('OOD check active — non-coffee leaves will be rejected')


Building coffee leaf feature reference...
Reference built from 45 coffee leaf images
  Similarity range : 0.2600 – 0.4217
  Mean similarity  : 0.3246
  OOD threshold    : 0.2100

OOD check active — non-coffee leaves will be rejected


## Step 4b — Coffee Region Database & Detection

In [8]:
# ── Coffee Region Database ────────────────────────────────────────────────────
COFFEE_REGIONS = [
    {
        "name": "Chikkamagaluru",
        "state": "Karnataka",
        "lat_min": 12.9, "lat_max": 13.5,
        "lon_min": 75.4, "lon_max": 76.2,
        "variety": "Arabica (Catimor, Caturra, S.795)",
        "soil_type": "Red laterite, forest loamy",
        "elevation": "900-1800m",
        "annual_rainfall": "1000-1500mm",
        "notes": "Birthplace of Indian coffee. Baba Budangiri Hills. Karnataka's largest coffee district."
    },
    {
        "name": "Kodagu (Coorg)",
        "state": "Karnataka",
        "lat_min": 11.9, "lat_max": 12.7,
        "lon_min": 75.4, "lon_max": 76.3,
        "variety": "Arabica (80%), Robusta (20%)",
        "soil_type": "Red and laterite loam, clay loam",
        "elevation": "600-1600m",
        "annual_rainfall": "1500-2500mm",
        "notes": "Largest coffee-producing district in India. Famous for shade-grown estate coffee."
    },
    {
        "name": "Hassan",
        "state": "Karnataka",
        "lat_min": 12.8, "lat_max": 13.3,
        "lon_min": 75.7, "lon_max": 76.5,
        "variety": "Arabica, Robusta",
        "soil_type": "Red laterite",
        "elevation": "800-1200m",
        "annual_rainfall": "900-1300mm",
        "notes": "Part of the Western Ghats coffee belt."
    },
    {
        "name": "Wayanad",
        "state": "Kerala",
        "lat_min": 11.3, "lat_max": 11.9,
        "lon_min": 75.7, "lon_max": 76.5,
        "variety": "Robusta (dominant), Arabica",
        "soil_type": "Laterite, red sandy loam",
        "elevation": "700-2100m",
        "annual_rainfall": "1500-3000mm",
        "notes": "Second largest coffee producer in India. Kerala's primary coffee district."
    },
    {
        "name": "Nilgiris",
        "state": "Tamil Nadu",
        "lat_min": 11.1, "lat_max": 11.7,
        "lon_min": 76.3, "lon_max": 76.9,
        "variety": "Arabica",
        "soil_type": "Clay loam, red loamy",
        "elevation": "1000-2500m",
        "annual_rainfall": "1400-2000mm",
        "notes": "High altitude specialty Arabica. Premium export quality."
    },
    {
        "name": "Araku Valley",
        "state": "Andhra Pradesh",
        "lat_min": 18.0, "lat_max": 18.5,
        "lon_min": 82.8, "lon_max": 83.5,
        "variety": "Arabica (tribal cultivation)",
        "soil_type": "Red sandy loam",
        "elevation": "900-1500m",
        "annual_rainfall": "1200-1600mm",
        "notes": "GI-tagged tribal coffee. Globally recognised specialty. Cultivated by Adivasi farmers."
    },
    {
        "name": "Yercaud / Salem Hills",
        "state": "Tamil Nadu",
        "lat_min": 11.6, "lat_max": 11.9,
        "lon_min": 78.0, "lon_max": 78.5,
        "variety": "Arabica",
        "soil_type": "Red loamy",
        "elevation": "1000-1500m",
        "annual_rainfall": "1000-1400mm",
        "notes": "Small but notable coffee hill region in Tamil Nadu."
    },
]

def detect_coffee_region(lat, lon):
    """
    Returns region dict if coordinates fall in a known coffee-growing area.
    Returns None if outside all known regions.
    """
    for r in COFFEE_REGIONS:
        if r["lat_min"] <= lat <= r["lat_max"] and r["lon_min"] <= lon <= r["lon_max"]:
            return r
    return None

def format_region_info(r):
    """Formats region details as a readable alert string."""
    return (
        f"COFFEE REGION IDENTIFIED\n"
        f"{'='*45}\n"
        f"  Region     : {r['name']}, {r['state']}\n"
        f"  Varieties  : {r['variety']}\n"
        f"  Soil type  : {r['soil_type']}\n"
        f"  Elevation  : {r['elevation']}\n"
        f"  Rainfall   : {r['annual_rainfall']}\n"
        f"  Notes      : {r['notes']}\n"
        f"{'='*45}"
    )

def format_non_coffee_warning(lat, lon):
    """Formats warning for non-coffee regions."""
    return (
        f"WARNING: Location Not a Known Coffee Region\n"
        f"{'='*45}\n"
        f"  Coordinates : lat={lat}, lon={lon}\n"
        f"  Status      : Outside all known South Indian\n"
        f"                coffee-growing regions\n"
        f"  Impact      : Soil readings from SoilGrids may\n"
        f"                not be representative for coffee.\n"
        f"                Regional defaults may not apply.\n"
        f"  Advice      : Verify location is correct or\n"
        f"                use Manual mode with lab readings.\n"
        f"{'='*45}"
    )

print("Coffee region database loaded")
print(f"Regions covered: {len(COFFEE_REGIONS)}")
for r in COFFEE_REGIONS:
    print(f"  {r['name']}, {r['state']}")


Coffee region database loaded
Regions covered: 7
  Chikkamagaluru, Karnataka
  Kodagu (Coorg), Karnataka
  Hassan, Karnataka
  Wayanad, Kerala
  Nilgiris, Tamil Nadu
  Araku Valley, Andhra Pradesh
  Yercaud / Salem Hills, Tamil Nadu


## Step 6 — Launch Gradio App with GEE Integration

In [9]:
DEFAULT_LAT = 13.3161
DEFAULT_LON = 75.7720
DEFAULTS = dict(N=120,P=45,K=200,pH=6.2,EC=0.8,OC=1.2,S=15,Zn=1.5,Fe=8.0,Cu=0.5,Mn=3.0,B=0.4)


def run_with_gee(img, lat, lon):
    lat, lon = float(lat), float(lon)

    # 1. Block non-coffee regions entirely
    region = detect_coffee_region(lat, lon)
    if region is None:
        block_msg = (
            "BLOCKED — Not a Coffee Growing Region\n"
            + "="*45 + "\n\n"
            + f"  Coordinates : lat={lat}, lon={lon}\n\n"
            + "  This location is outside all known South\n"
            + "  Indian coffee-growing regions.\n\n"
            + "  This system only supports coffee crop\n"
            + "  health monitoring.\n\n"
            + "  Supported regions:\n"
            + "    Chikkamagaluru, Kodagu, Hassan (Karnataka)\n"
            + "    Wayanad (Kerala)\n"
            + "    Nilgiris, Yercaud (Tamil Nadu)\n"
            + "    Araku Valley (Andhra Pradesh)\n\n"
            + "  Please use coordinates within a known coffee\n"
            + "  growing region or use Mode 2 (Manual).\n"
            + "="*45
        )
        return gr.update(value=block_msg, visible=True), "", gr.update()

    region_msg = format_region_info(region)

    # 2. Check leaf BEFORE fetching soil — no point fetching if leaf is invalid
    leaf_name, leaf_conf, top3, leaf_probs = predict_leaf(img)

    if leaf_name is None:
        return gr.update(value=region_msg, visible=True), "Please upload a leaf image.", gr.update()

    if leaf_name == 'NOT_COFFEE_LEAF':
        rejection = build_report(None, None, None, None,
                                 leaf_name, leaf_conf, None, None)
        return gr.update(value=region_msg, visible=True), rejection, gr.update()

    # 3. Leaf is valid — now fetch soil
    try:
        readings, sources = fetch_soil_from_gee(lat, lon)
    except Exception as e:
        return gr.update(value=region_msg, visible=True), f"Soil fetch error: {str(e)}. Use Manual mode.", gr.update()

    # 4. Run soil agent + fusion
    soil_cls, soil_label, soil_conf, soil_probs = predict_soil(readings)
    readings_str = "\n".join(f"  {k:<4}: {v}" for k,v in readings.items())
    report = build_report(soil_cls, soil_label, soil_conf, soil_probs,
                          leaf_name, leaf_conf, top3, leaf_probs, sources)
    full_report = f"FETCHED SOIL READINGS (lat={lat}, lon={lon})\n{readings_str}\n\n{report}"
    return gr.update(value=region_msg, visible=True), full_report, gr.update()




    readings, sources = fetch_soil_from_gee(lat, lon)

def run_manual(img, N,P,K,pH,EC,OC,S,Zn,Fe,Cu,Mn,B):
    leaf_name, leaf_conf, top3, leaf_probs = predict_leaf(img)
    if leaf_name is None:
        return gr.update(visible=False), "Please upload a leaf image.", gr.update()
    if leaf_name == 'NOT_COFFEE_LEAF':
        rejection = build_report(None, None, None, None,
                                 leaf_name, leaf_conf, None, None)
        return gr.update(visible=False), rejection, gr.update()
    readings = dict(N=N,P=P,K=K,pH=pH,EC=EC,OC=OC,S=S,Zn=Zn,Fe=Fe,Cu=Cu,Mn=Mn,B=B)
    soil_cls, soil_label, soil_conf, soil_probs = predict_soil(readings)
    report = build_report(soil_cls, soil_label, soil_conf, soil_probs,
                          leaf_name, leaf_conf, top3, leaf_probs)
    return gr.update(visible=False), report, gr.update()


with gr.Blocks(title="Coffee Crop Health Monitor", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # Coffee Crop Health Monitor
    **Mode 1:** Enter GPS coordinates — soil auto-fetched from SoilGrids (South India coffee regions only).
    **Mode 2:** Enter soil readings manually.
    Swin Transformer (97.48% test acc) + XGBoost (~90%) + SoilGrids REST API.
    """)

    with gr.Row():
        img_input = gr.Image(label="Upload leaf photo", type="numpy", height=260, scale=1)

        with gr.Column(scale=2):
            with gr.Tab("Mode 1 — Auto (GEE)"):
                gr.Markdown("GPS coordinates of the farm. Only South Indian coffee-growing regions accepted.")
                with gr.Row():
                    lat = gr.Number(label="Latitude",  value=DEFAULT_LAT, precision=4)
                    lon = gr.Number(label="Longitude", value=DEFAULT_LON, precision=4)
                gr.Markdown("*Default: Chikkamagaluru, Karnataka*")
                btn_gee = gr.Button("Fetch Soil + Assess", variant="primary")

            with gr.Tab("Mode 2 — Manual"):
                gr.Markdown("Enter soil readings from a lab test or soil sensor.")
                with gr.Row():
                    N  = gr.Number(label="N",  value=DEFAULTS["N"],  precision=1)
                    P  = gr.Number(label="P",  value=DEFAULTS["P"],  precision=1)
                    K  = gr.Number(label="K",  value=DEFAULTS["K"],  precision=1)
                with gr.Row():
                    pH = gr.Number(label="pH", value=DEFAULTS["pH"], precision=2)
                    EC = gr.Number(label="EC", value=DEFAULTS["EC"], precision=2)
                    OC = gr.Number(label="OC", value=DEFAULTS["OC"], precision=2)
                with gr.Row():
                    S  = gr.Number(label="S",  value=DEFAULTS["S"],  precision=1)
                    Zn = gr.Number(label="Zn", value=DEFAULTS["Zn"], precision=2)
                    Fe = gr.Number(label="Fe", value=DEFAULTS["Fe"], precision=1)
                with gr.Row():
                    Cu = gr.Number(label="Cu", value=DEFAULTS["Cu"], precision=2)
                    Mn = gr.Number(label="Mn", value=DEFAULTS["Mn"], precision=1)
                    B  = gr.Number(label="B",  value=DEFAULTS["B"],  precision=2)
                btn_manual = gr.Button("Run Assessment", variant="primary")

    region_alert = gr.Textbox(label="Location Status", lines=9, visible=False,
                               show_copy_button=False, interactive=False)
    output = gr.Textbox(label="Health Assessment Report", lines=22, show_copy_button=True)

    gr.Markdown("---")
    gr.Markdown("*Satellite: N, pH, OC from SoilGrids REST API (250m). Defaults: P, K, EC, S, Zn, Fe, Cu, Mn, B (Karnataka averages).*")

    btn_gee.click(fn=run_with_gee, inputs=[img_input, lat, lon],
                  outputs=[region_alert, output, region_alert])
    btn_manual.click(fn=run_manual, inputs=[img_input,N,P,K,pH,EC,OC,S,Zn,Fe,Cu,Mn,B],
                     outputs=[region_alert, output, region_alert])

demo.launch(share=True, debug=False)


/tmp/ipykernel_3492/3090489636.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Coffee Crop Health Monitor", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94b4479ab422da67ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
